# AI-Powered Robo Advisor for Wealth Management
## ML Trading Bot — Full Pipeline
**Course:** Build AI with AWS | **Week:** 3–5 | **Date:** June 2026

---

### Pipeline Overview
1. **Data Collection** — Yahoo Finance OHLCV + Fundamentals (Top 50 S&P 500)
2. **Feature Engineering** — Returns, Moving Averages, Volatility, PE/EPS
3. **Model Training** — Logistic Regression vs Random Forest
4. **Model Evaluation** — Accuracy, F1, Classification Report
5. **Trading Strategy** — Signal generation + risk-level portfolio construction
6. **Backtesting** — ML Strategy vs Buy & Hold benchmark
7. **Visualization** — Cumulative returns chart saved as PNG

## Cell 1 — Install Dependencies

In [ ]:
# Run this cell first (Google Colab / SageMaker)
!pip install yfinance scikit-learn pandas numpy matplotlib seaborn --quiet

## Cell 2 — Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless rendering for SageMaker/Colab
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics         import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)

print('All imports successful ✓')

## Cell 3 — Configuration

In [ ]:
# Top 50 S&P 500 companies by market capitalization
TICKERS = [
    'AAPL','MSFT','NVDA','AMZN','GOOGL','META','JPM','V',
    'UNH', 'TSLA','AVGO','LLY', 'HD',   'MRK', 'ABBV','PG',
    'JNJ', 'XOM', 'MA',  'BAC', 'CVX',  'COST','PEP', 'TMO',
    'ADBE','ACN', 'CRM', 'NFLX','AMD',  'TXN', 'QCOM','HON',
    'AMGN','SBUX','GE',  'MS',  'BLK',  'ISRG','MDT', 'GILD',
    'CME', 'ZTS', 'REGN','MO',  'ADP',  'ITW', 'CI',  'MMC',
    'VRTX','EOG'
]

TRAIN_START = '2019-01-01'
TRAIN_END   = '2022-12-31'
TEST_START  = '2022-01-01'
TEST_END    = '2024-12-31'

# Risk level definitions
RISK_MAP = {'low': 5, 'medium': 15, 'high': 25}

print(f'Configuration loaded — {len(TICKERS)} tickers')
print(f'Train: {TRAIN_START} → {TRAIN_END}')
print(f'Test:  {TEST_START} → {TEST_END}')

## Cell 4 — Data Collection (Yahoo Finance OHLCV)

In [ ]:
print('Downloading OHLCV data from Yahoo Finance...')

raw = yf.download(
    TICKERS,
    start=TRAIN_START,
    end=TEST_END,
    auto_adjust=True,
    progress=True
)

close  = raw['Close']
volume = raw['Volume']

print(f'\nData shape: {close.shape}')
print(f'Date range: {close.index[0].date()} → {close.index[-1].date()}')
print(f'Missing values per ticker (top 5):')
print(close.isnull().sum().sort_values(ascending=False).head())

## Cell 5 — Feature Engineering

In [ ]:
def build_features(close_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build ML features for each ticker:
    - 1/5/20-day returns
    - 5/20-day MA ratios (price relative to moving average)
    - 20-day rolling volatility
    - Binary target: next-day return positive (1) or negative (0)
    """
    feature_frames = []

    for ticker in close_df.columns:
        s = close_df[ticker].dropna()
        if len(s) < 30:
            continue

        df = pd.DataFrame(index=s.index)
        df['ticker']    = ticker
        df['ret_1d']    = s.pct_change(1)
        df['ret_5d']    = s.pct_change(5)
        df['ret_20d']   = s.pct_change(20)
        df['ma5_ratio'] = s / s.rolling(5).mean()  - 1  # price vs 5-day MA
        df['ma20_ratio']= s / s.rolling(20).mean() - 1  # price vs 20-day MA
        df['vol_20d']   = df['ret_1d'].rolling(20).std() # 20-day realized volatility
        # Target: next-day direction (1 = up, 0 = down/flat)
        df['target']    = (s.shift(-1) > s).astype(int)

        feature_frames.append(df)

    all_features = pd.concat(feature_frames).dropna()
    return all_features


features_df = build_features(close)

print(f'Feature matrix shape: {features_df.shape}')
print(f'Features: {[c for c in features_df.columns if c not in ["ticker","target"]]}')
print(f'\nClass balance:')
print(features_df['target'].value_counts(normalize=True).round(3))

## Cell 6 — Train/Test Split

In [ ]:
FEATURE_COLS = ['ret_1d','ret_5d','ret_20d','ma5_ratio','ma20_ratio','vol_20d']

train_df = features_df[features_df.index <= TRAIN_END]
test_df  = features_df[features_df.index > TRAIN_END]

X_train = train_df[FEATURE_COLS].values
y_train = train_df['target'].values
X_test  = test_df[FEATURE_COLS].values
y_test  = test_df['target'].values

# Scale features
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train samples: {len(X_train):,}')
print(f'Test  samples: {len(X_test):,}')

## Cell 7 — Model 1: Logistic Regression

In [ ]:
print('Training Logistic Regression...')

lr_model = LogisticRegression(max_iter=500, random_state=42, C=1.0)
lr_model.fit(X_train, y_train)

lr_preds  = lr_model.predict(X_test)
lr_acc    = accuracy_score(y_test, lr_preds)
lr_f1     = f1_score(y_test, lr_preds, average='weighted')

print(f'\nLogistic Regression Results:')
print(f'  Test Accuracy : {lr_acc:.4f} ({lr_acc*100:.1f}%)')
print(f'  Weighted F1   : {lr_f1:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, lr_preds, target_names=['DOWN','UP']))

## Cell 8 — Model 2: Random Forest (Selected Model)

In [ ]:
print('Training Random Forest (100 estimators, max_depth=6)...')

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_acc   = accuracy_score(y_test, rf_preds)
rf_f1    = f1_score(y_test, rf_preds, average='weighted')

print(f'\nRandom Forest Results (SELECTED MODEL):')
print(f'  Test Accuracy : {rf_acc:.4f} ({rf_acc*100:.1f}%)')
print(f'  Weighted F1   : {rf_f1:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, rf_preds, target_names=['DOWN','UP']))

print(f'\nTop 5 Feature Importances:')
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
print(importances.sort_values(ascending=False).head())

## Cell 9 — Model Comparison Chart (Screenshot A)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

models  = ['Logistic Regression', 'Random Forest (Selected)']
accs    = [lr_acc * 100, rf_acc * 100]
colors  = ['#E57373', '#1565C0']

bars = ax.bar(models, accs, color=colors, width=0.45, edgecolor='white', linewidth=1.5)

# Baseline line
ax.axhline(50, color='#9E9E9E', linestyle='--', linewidth=1.2, label='Baseline (50%)')

# Value labels on bars
for bar, acc in zip(bars, accs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{acc:.1f}%',
        ha='center', va='bottom', fontsize=13, fontweight='bold', color='#1A1A1A'
    )

ax.set_title('ML Model Comparison: Logistic Regression vs Random Forest',
             fontsize=14, fontweight='bold', pad=14)
ax.set_ylabel('Test accuracy (%)', fontsize=11)
ax.set_ylim(48, 60)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.tick_params(labelsize=11)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', color='#E0E0E0', linewidth=0.8)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

## Cell 10 — Trading Strategy & Backtesting

In [ ]:
def backtest_strategy(close_df, model, scaler, feature_cols, test_start, risk='medium'):
    """
    Backtest the ML trading strategy vs equal-weight buy & hold.

    Strategy:
      - Each day: predict direction for all tickers
      - Long the top-N BUY-signal stocks (N = risk level)
      - Equal weight allocation, daily rebalancing
    """
    risk_n = RISK_MAP[risk]
    test_close = close_df[close_df.index >= test_start].copy()
    daily_returns = test_close.pct_change().fillna(0)

    ml_daily   = []
    bh_daily   = []
    dates      = []

    for i in range(1, len(test_close)):
        date = test_close.index[i]
        prev = test_close.iloc[i-1]

        # Build feature vector for each ticker on this day
        scores = {}
        for ticker in close_df.columns:
            try:
                s = close_df[ticker].iloc[:close_df.index.get_loc(date)]
                if len(s) < 21:
                    continue
                feats = np.array([
                    s.pct_change(1).iloc[-1],
                    s.pct_change(5).iloc[-1],
                    s.pct_change(20).iloc[-1],
                    s.iloc[-1] / s.rolling(5).mean().iloc[-1]  - 1,
                    s.iloc[-1] / s.rolling(20).mean().iloc[-1] - 1,
                    s.pct_change(1).rolling(20).std().iloc[-1]
                ]).reshape(1, -1)
                feats = scaler.transform(feats)
                prob  = model.predict_proba(feats)[0][1]  # P(up)
                scores[ticker] = prob
            except Exception:
                continue

        # Select top-N by predicted probability
        top_n   = sorted(scores, key=scores.get, reverse=True)[:risk_n]
        ml_ret  = daily_returns.loc[date, top_n].mean() if top_n else 0.0
        bh_ret  = daily_returns.loc[date].mean()

        ml_daily.append(ml_ret)
        bh_daily.append(bh_ret)
        dates.append(date)

    ml_cum = (1 + pd.Series(ml_daily, index=dates)).cumprod()
    bh_cum = (1 + pd.Series(bh_daily, index=dates)).cumprod()

    return ml_cum, bh_cum


print('Running backtest (medium risk, 15 stocks)...')
print('Note: Full backtest over 2022-2024 may take 2-4 minutes...')

# For speed in demo, use a representative date range
ml_cum, bh_cum = backtest_strategy(
    close, rf_model, scaler, FEATURE_COLS,
    test_start=TEST_START, risk='medium'
)

final_ml = (ml_cum.iloc[-1] - 1) * 100
final_bh = (bh_cum.iloc[-1] - 1) * 100
alpha    = final_ml - final_bh

print(f'\nBacktest Results:')
print(f'  ML Strategy Return  : {final_ml:+.1f}%')
print(f'  Buy & Hold Return   : {final_bh:+.1f}%')
print(f'  Alpha Generated     : {alpha:+.1f}%')

## Cell 11 — Cumulative Returns Chart (Screenshot B)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

ax.plot(ml_cum.index, ml_cum.values,
        color='#1565C0', linewidth=2.0, label='ML Strategy (Random Forest)')
ax.plot(bh_cum.index, bh_cum.values,
        color='#9E9E9E', linewidth=1.5, linestyle='--', label='Buy & Hold Benchmark')

# Shade outperformance
ax.fill_between(ml_cum.index, ml_cum.values, bh_cum.values,
                where=(ml_cum.values > bh_cum.values),
                alpha=0.15, color='#1565C0', label='Outperformance')

# Annotations
ax.annotate(
    f'ML: {final_ml:+.1f}%',
    xy=(ml_cum.index[-1], ml_cum.values[-1]),
    xytext=(-80, 10), textcoords='offset points',
    fontsize=10, fontweight='bold', color='#1565C0',
    arrowprops=dict(arrowstyle='->', color='#1565C0')
)
ax.annotate(
    f'B&H: {final_bh:+.1f}%',
    xy=(bh_cum.index[-1], bh_cum.values[-1]),
    xytext=(-80, -20), textcoords='offset points',
    fontsize=10, color='#757575',
    arrowprops=dict(arrowstyle='->', color='#757575')
)

ax.set_title('Cumulative Returns: ML Trading Strategy vs Buy & Hold (2022-2024)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Portfolio value (normalised to 1.0)', fontsize=11)
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.spines[['top','right']].set_visible(False)
ax.grid(color='#E0E0E0', linewidth=0.7, alpha=0.8)
ax.legend(fontsize=10, loc='upper left')

plt.tight_layout()
plt.savefig('cumulative_returns.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cumulative_returns.png')

## Cell 12 — Sample Portfolio Output

In [ ]:
def build_sample_portfolio(close_df, model, scaler, feature_cols, risk, amount):
    """Build a sample portfolio using the latest available data."""
    risk_n  = RISK_MAP[risk]
    latest  = close_df.iloc[-1]
    scores  = {}

    for ticker in close_df.columns:
        try:
            s = close_df[ticker].dropna()
            if len(s) < 21:
                continue
            feats = np.array([
                s.pct_change(1).iloc[-1],
                s.pct_change(5).iloc[-1],
                s.pct_change(20).iloc[-1],
                s.iloc[-1] / s.rolling(5).mean().iloc[-1]  - 1,
                s.iloc[-1] / s.rolling(20).mean().iloc[-1] - 1,
                s.pct_change(1).rolling(20).std().iloc[-1]
            ]).reshape(1, -1)
            feats = scaler.transform(feats)
            prob  = model.predict_proba(feats)[0][1]
            scores[ticker] = prob
        except:
            continue

    top_n  = sorted(scores, key=scores.get, reverse=True)[:risk_n]
    weight = amount / len(top_n)
    return top_n, weight


for risk_level in ['low', 'medium', 'high']:
    stocks, w = build_sample_portfolio(close, rf_model, scaler, FEATURE_COLS, risk_level, 10000)
    print(f'\n{risk_level.upper()} Risk Portfolio (${w*len(stocks):,.0f}, {len(stocks)} stocks, ${w:,.2f}/stock):')
    print(' | '.join([f'{s}: ${w:,.2f}' for s in stocks]))

## Cell 13 — Summary Statistics

In [ ]:
print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'Dataset:     Top 50 S&P 500 | {TRAIN_START} → {TEST_END}')
print(f'Train:       {TRAIN_START} → {TRAIN_END}')
print(f'Test:        {TEST_START} → {TEST_END}')
print()
print(f'Model 1 — Logistic Regression')
print(f'  Accuracy:  {lr_acc*100:.1f}%')
print(f'  F1 Score:  {lr_f1:.4f}')
print()
print(f'Model 2 — Random Forest (SELECTED)')
print(f'  Accuracy:  {rf_acc*100:.1f}%')
print(f'  F1 Score:  {rf_f1:.4f}')
print()
print(f'Backtest Performance (Medium Risk, 15 stocks):')
print(f'  ML Strategy Return  : {final_ml:+.1f}%')
print(f'  Buy & Hold Return   : {final_bh:+.1f}%')
print(f'  Alpha Generated     : {alpha:+.1f}%')
print()
print('Output files:')
print('  cumulative_returns.png  — backtest chart')
print('  model_comparison.png    — accuracy comparison')
print('=' * 60)